# Hierarchical Bayesian Model — Sector Automation Risk

The default `ScenarioSimulator` treats each labour sector's automation risk as an **independent** truncated normal centred on the McKinsey/WEF/OECD consensus. That's a defensible choice but it discards a known fact: uncertainty about *"how automatable is the 2040 economy"* is partly a global property, not eleven independent properties.

This notebook fits a small hierarchical Bayesian model (PyMC + NUTS) that **partially pools** the sector estimates toward a global mean, with a sector-level deviation. Sectors with wide reported uncertainty get shrunk more aggressively toward the global mean — a textbook regularisation guard against over-fitting the consensus numbers.

**Model**

$$
\begin{aligned}
\mu_g &\sim \mathcal{N}(0.5,\,0.2) \\
\tau &\sim \text{HalfNormal}(0.15) \\
z_s &\sim \mathcal{N}(0,\,1) \quad \text{(non-centred parameterisation)} \\
\theta_s &= \text{clip}(\mu_g + \tau \cdot z_s,\, 0.01,\, 0.99) \\
y_s &\sim \mathcal{N}(\theta_s,\, \sigma_s^{\text{reported}})
\end{aligned}
$$

Implementation: [`src/models/bayesian.py`](../src/models/bayesian.py).

**Why this is portfolio-worthy**
- Non-centred (`z_s`) parameterisation — known fix for Neal's funnel pathology.
- ArviZ-compatible `InferenceData` so convergence diagnostics (`r_hat`, `ess`) are one line away.
- `posterior_sector_risks(n)` is a drop-in replacement for the simulator's truncnorm draws — the whole unemployment model gains a Bayesian backbone for free.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.models.bayesian import build_sector_risk_model, fit_sector_risk_model
from src.models.scenarios import SECTORS

## 1. Sample the posterior

Modest sample budget for a clean notebook (4 chains × 1000 draws after 1000 tuning steps). Production runs would use `target_accept=0.99` and `draws=4000`.

In [ ]:
result = fit_sector_risk_model(
    draws=1000, tune=1000, chains=4,
    target_accept=0.95, random_seed=42, progressbar=True,
)
result.idata

## 2. Convergence diagnostics

All `r_hat` should be ≤ 1.01 and `ess_bulk` should be in the thousands. Sampling pathologies show up here first.

In [ ]:
az.summary(result.idata, var_names=['mu_global', 'tau', 'theta'], round_to=3)

In [ ]:
az.plot_trace(result.idata, var_names=['mu_global', 'tau'], figsize=(11, 4))
plt.tight_layout()
plt.show()

## 3. Posterior sector ranking

Because of partial pooling, sectors with very wide consensus uncertainty get shrunk toward the global mean. Compare each sector's posterior mean to its prior mean (the published consensus) to see how much pooling moved it.

In [ ]:
summary = result.summary().reset_index(drop=True)
prior = pd.DataFrame([{'sector': s.name, 'prior_mean': s.risk_mean, 'prior_std': s.risk_std} for s in SECTORS])
comparison = summary.merge(prior, on='sector')
comparison['shrinkage_pp'] = (comparison['posterior_mean'] - comparison['prior_mean']) * 100
comparison.sort_values('posterior_mean', ascending=False)

In [ ]:
az.plot_forest(result.idata, var_names=['theta'], combined=True, figsize=(8, 6))
plt.title('Posterior 94% HDI per sector — partial pooling regularises tails')
plt.tight_layout()
plt.show()

## 4. Drop-in replacement for the Monte Carlo simulator

`posterior_sector_risks(n)` returns `(n, n_sectors)` — exactly the shape the vectorised `ScenarioSimulator._sample_sector_risks_matrix` returns. Swap one for the other and the rest of the unemployment pipeline runs unchanged with a fully Bayesian backbone.

In [ ]:
draws = result.posterior_sector_risks(n_samples=5000, seed=42)
print('Shape:', draws.shape)
print('First 3 draws (sector x):')
pd.DataFrame(draws[:3], columns=result.sector_names).round(3)

## 5. Posterior predictive vs frequentist truncnorm

Quick sanity plot: for one sector, compare the Bayesian posterior of `θ` against the original truncated-normal prior. The Bayesian posterior should be tighter and centred slightly differently when pooling has moved the sector toward the global mean.

In [ ]:
from scipy.stats import truncnorm

sector_idx = 0  # Administrative Services — high consensus, narrow prior
sector = SECTORS[sector_idx]
a = (0 - sector.risk_mean) / sector.risk_std
b = (1 - sector.risk_mean) / sector.risk_std
frequentist = truncnorm.rvs(a, b, loc=sector.risk_mean, scale=sector.risk_std,
                            size=5000, random_state=42)
bayesian = result.posterior_theta[:, sector_idx]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(frequentist, bins=40, alpha=0.5, label='Truncated normal (current)', color='#58a6ff')
ax.hist(bayesian,    bins=40, alpha=0.5, label='Bayesian posterior',         color='#d2a8ff')
ax.set_xlabel(f'Automation risk — {sector.name}')
ax.set_ylabel('Density')
ax.set_title('Frequentist truncnorm vs hierarchical Bayesian posterior')
ax.legend()
plt.tight_layout()
plt.show()